---
title: LayerNorm, GELU, and residuals
description: Three supporting components that make stacking transformer blocks stable — layer normalization keeps activations well-scaled, GELU provides a smooth nonlinearity, and residual connections prevent gradients from vanishing in deep networks.
---

The multi-head attention block from the last episode is powerful, but three essential
supporting components must surround it before we can stack 12 of them:

1. **LayerNorm** — normalizes each token vector to zero mean, unit variance, then
   scales and shifts with learned parameters. Prevents activation blow-up or collapse as
   signals pass through many layers.
2. **GELU** — a smooth, differentiable activation that replaces ReLU in GPT-style
   transformers. Its "soft" gate behavior near zero provides better gradient flow.
3. **Residual connections** — add the block's input directly to its output, giving
   gradients a highway that skips over entire blocks. Critical for training 12, 24, or
   96 layers deep.

## LayerNorm from scratch



In [ ]:
import torch
import torch.nn as nn

torch.manual_seed(123)
batch_example = torch.randn(2, 5)
layer = nn.Sequential(nn.Linear(5, 6), nn.ReLU())
out = layer(batch_example)
print(out)
# [[0.226, 0.347, 0.000, 0.222, 0.000, 0.000],
#  [0.213, 0.239, 0.000, 0.520, 0.330, 0.000]]



The mean and variance differ between examples — and will differ layer to layer as the
network trains. This is the instability LayerNorm fixes.



In [ ]:
# Mean and variance per row (per token in the GPT setting)
mean = out.mean(dim=-1, keepdim=True)
var  = out.var(dim=-1, keepdim=True, unbiased=False)
print("mean:", mean)
print("var:", var)

# Manual normalization: subtract mean, divide by std
norm_out = (out - mean) / (var + 1e-5).sqrt()
print("Row 0 mean after norm:", norm_out[0].mean().item())   # ≈ 0
print("Row 0 std after norm:", norm_out[0].std().item())     # ≈ 1



The real `LayerNorm` class adds two trainable parameters: `scale` (γ) and `shift` (β),
both initialized to 1 and 0 respectively. These let the model undo the normalization if
that's what the gradient says is best:



In [ ]:
class LayerNorm(nn.Module):
    def __init__(self, emb_dim):
        super().__init__()
        self.eps   = 1e-5
        self.scale = nn.Parameter(torch.ones(emb_dim))   # γ, learnable
        self.shift = nn.Parameter(torch.zeros(emb_dim))  # β, learnable

    def forward(self, x):
        mean = x.mean(dim=-1, keepdim=True)
        var  = x.var(dim=-1, keepdim=True, unbiased=False)
        norm_x = (x - mean) / (var + self.eps).sqrt()
        return self.scale * norm_x + self.shift



The `unbiased=False` flag uses `N` (not `N-1`) in the denominator — consistent with the
"biased" estimator the original PyTorch `nn.LayerNorm` uses internally. Using `N-1`
here would give slightly different scale behavior during training.

Here's what layer norm does to a row of numbers visually:

export const rawRow = [[0.226, 0.347, 0.000, 0.222, 0.000, 0.000]]
export const normRow = [[1.23, 1.87, -0.98, 1.20, -0.98, -0.98]]

<Scene autoPlayMs={2500}>
  <Step caption="Raw activations after ReLU — arbitrary mean and spread, some dead at 0.">
    <Matrix id="ln-viz" values={rawRow} colLabels={['f0','f1','f2','f3','f4','f5']} colorScale="sequential" precision={3} cellSize={60} />
  </Step>
  <Step caption="After LayerNorm — zero mean, unit variance. The model can now reliably propagate gradients.">
    <Matrix id="ln-viz" values={normRow} colLabels={['f0','f1','f2','f3','f4','f5']} colorScale="diverging" precision={2} cellSize={60} />
  </Step>
</Scene>

## GELU — a smoother ReLU

GPT-2 uses **GELU** (Gaussian Error Linear Unit) instead of ReLU. Where ReLU is a hard
gate (negative → 0, positive → unchanged), GELU is a soft, smooth gate that blends the
identity and zero continuously:



In [ ]:
import math

class GELU(nn.Module):
    def forward(self, x):
        return 0.5 * x * (1 + torch.tanh(
            math.sqrt(2.0 / math.pi) * (x + 0.044715 * x**3)
        ))



export const geluIn  = [[-2.0], [-1.5], [-1.0], [-0.5], [0.0], [0.5], [1.0], [1.5], [2.0]]
export const geluOut = [[-0.05], [-0.10], [-0.16], [-0.15], [0.0], [0.35], [0.84], [1.40], [1.95]]
export const reluOut = [[0.0], [0.0], [0.0], [0.0], [0.0], [0.5], [1.0], [1.5], [2.0]]

<Scene autoPlayMs={2000}>
  <Step caption="ReLU: hard cutoff at 0 — values below 0 are dead, gradients are exactly 0 there.">
    <Matrix id="activation" values={reluOut} rowLabels={['-2','-1.5','-1','-0.5','0','0.5','1','1.5','2']} colLabels={["ReLU(x)"]} colorScale="diverging" precision={2} cellSize={52} />
  </Step>
  <Step caption="GELU: smooth — slightly negative near 0, gradually recovering. Better gradient flow through near-zero activations.">
    <Matrix id="activation" values={geluOut} rowLabels={['-2','-1.5','-1','-0.5','0','0.5','1','1.5','2']} colLabels={["GELU(x)"]} colorScale="diverging" precision={2} cellSize={52} />
  </Step>
</Scene>

## FeedForward — the position-wise network

Each transformer block runs a two-layer MLP on every token position independently:



In [ ]:
class FeedForward(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(cfg["emb_dim"], 4 * cfg["emb_dim"]),  # expand ×4
            GELU(),
            nn.Linear(4 * cfg["emb_dim"], cfg["emb_dim"]),  # contract back
        )

    def forward(self, x):
        return self.layers(x)



The expand-then-contract pattern (`768 → 3072 → 768` in GPT-2 small) gives the network
a wide intermediate space to perform token-level transformations. Each token's vector is
processed *independently* here — no cross-token interaction like attention. The linear
layers mix features *within* a token; attention mixes *across* tokens. Together they
complement each other.

## Residual connections — the gradient highway



In [ ]:
# Residual shortcut: output = block(x) + x
class ExampleResidualBlock(nn.Module):
    def __init__(self, emb_dim):
        super().__init__()
        self.linear = nn.Linear(emb_dim, emb_dim)
        self.gelu   = GELU()

    def forward(self, x):
        return x + self.gelu(self.linear(x))   # ← shortcut: add input back



Without residuals, gradients must flow through every transformation in sequence.
In a 12-layer network each backprop step multiplies the gradient by the derivative
of every layer — small derivatives compound multiplicatively and the gradient
shrinks toward zero ("vanishes") before reaching the early layers.

With residuals, the gradient also flows backward through the `+` operation, which
passes it through unchanged to the input of the block — providing a direct highway
that bypasses every transformation. Early layers always receive a strong gradient.

export const noRes = [[0.8], [0.32], [0.13], [0.05], [0.02], [0.01], [0.002], [0.001], [0.0003], [0.0001], [0.00003], [0.00001]]
export const withRes = [[0.8], [0.75], [0.71], [0.68], [0.65], [0.62], [0.60], [0.58], [0.56], [0.54], [0.52], [0.50]]

<Scene autoPlayMs={2500}>
  <Step caption="Without residuals — gradient halves each layer, near-zero after 12 layers. Early weights barely update.">
    <Matrix id="grad-flow" values={noRes} rowLabels={['L1','L2','L3','L4','L5','L6','L7','L8','L9','L10','L11','L12']} colLabels={["grad"]} colorScale="sequential" precision={4} />
  </Step>
  <Step caption="With residuals — gradient flows through the shortcut, stays strong all the way to layer 1.">
    <Matrix id="grad-flow" values={withRes} rowLabels={['L1','L2','L3','L4','L5','L6','L7','L8','L9','L10','L11','L12']} colLabels={["grad"]} colorScale="sequential" precision={2} />
  </Step>
</Scene>

GPT-2 uses **pre-layer-norm** (apply LayerNorm *before* each sublayer, not after), which
further stabilizes training vs the original transformer paper's post-norm variant. Both
are valid; modern models almost universally prefer pre-norm.

Next: [Assembling GPT](/series/08-assembling-gpt).
